# Attention / Mamba encoder-decoder experiments on Kaggle

Train the four scratch variants, retaining shared decoder cross-attention.

Enable **Internet** and a **GPU accelerator** in Kaggle settings. Edit `REPO_URL`
below after pushing this project to GitHub. The workflow uses GPU 0, even if
Kaggle exposes two GPUs. Long implementations live in `src/qa_assignment/`.

Artifacts go to `/kaggle/working/qa_assignment/`. Each variant and seed has its
own checkpoint directory. Kaggle working files are temporary: download the
archive or save the notebook outputs, then attach those outputs to resume a
later session. Checkpoints are written every 250 optimizer steps and each epoch.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/NhanPhan090806/Mamba_Attention"  # fill in your GitHub repository URL after pushing the project
REPO_REF = "main"  # preferably the same commit SHA in BOTH notebooks
SOURCE_SUBDIR = "."  # edit if this assignment lives below the repository root
WORKSPACE = Path("/kaggle/working")
CLONE_ROOT = WORKSPACE / "qa_assignment_source"
OUTPUT_ROOT = WORKSPACE / "qa_assignment"
RESTORE_FROM = None  # e.g. Path('/kaggle/input/prior-qa-output/qa_assignment')

if not REPO_URL.strip():
    raise ValueError("Set REPO_URL to the GitHub repository you pushed this project to.")
if not WORKSPACE.is_dir():
    raise RuntimeError("This notebook is intended for Kaggle. Run local tests with pytest.")
if not (CLONE_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", REPO_URL, str(CLONE_ROOT)], check=True)
subprocess.run(["git", "-C", str(CLONE_ROOT), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(CLONE_ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
REPO_ROOT = (CLONE_ROOT / SOURCE_SUBDIR).resolve()
assert (REPO_ROOT / "pyproject.toml").is_file(), "Check SOURCE_SUBDIR."
print("Source:", REPO_ROOT)
print("Artifacts:", OUTPUT_ROOT)
subprocess.run(["git", "-C", str(CLONE_ROOT), "rev-parse", "HEAD"], check=True)

## Install the Python helpers

Run setup before importing project modules. These requirements pin the Hugging
Face interface used here and preserve Kaggle's preinstalled CUDA-enabled PyTorch.
Setup also adds the cloned `src/` directory to this running kernel's import path;
a subprocess editable install alone cannot refresh an already-started kernel.
If you previously imported Transformers in this session, restart the session
after installing and rerun from the top. A successful import is followed by an
actual forward/backward and generation probe before training.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_ROOT / "requirements-kaggle.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "-e", str(REPO_ROOT)], check=True)

# Editable-install .pth files are read at interpreter startup. This kernel
# predates the pip subprocess, so explicitly expose the cloned sources now.
import importlib

SOURCE_ROOT = (REPO_ROOT / "src").resolve()
if not (SOURCE_ROOT / "qa_assignment" / "__init__.py").is_file():
    raise FileNotFoundError(f"Missing qa_assignment sources under {SOURCE_ROOT}; check SOURCE_SUBDIR.")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()

import qa_assignment
if Path(qa_assignment.__file__).resolve().parent != SOURCE_ROOT / "qa_assignment":
    raise RuntimeError("qa_assignment was loaded from another checkout. Restart the session and rerun setup.")
print("Loaded helpers:", qa_assignment.__file__)

import torch
from qa_assignment.utils import environment_info
from qa_assignment.runtime import restore_artifacts

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU accelerator in Kaggle settings.")
DEVICE = "cuda:0"
if RESTORE_FROM is not None:
    print(restore_artifacts(RESTORE_FROM, OUTPUT_ROOT))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
environment_info(REPO_ROOT)

## Mamba CUDA setup

Install the pinned Mamba-1 implementation against Kaggle's existing PyTorch.
This may download a matching binary wheel or compile an extension; setup time
is excluded from model benchmarks. The helper requires real selective-scan and
causal-convolution CUDA extensions. It never substitutes a slow Python model.
Select a modern GPU runtime; on older accelerators a Triton/kernel probe can
fail. In that case stop and select a compatible runtime, rather than reporting
fallback timing as Mamba efficiency. See README for troubleshooting.

In [ ]:
from qa_assignment.runtime import install_mamba, require_mamba_kernels

INSTALL_MAMBA = True
if INSTALL_MAMBA:
    install_mamba()
print(require_mamba_kernels())

## Download the whole dataset and prepare identical splits

Both official SQuAD 1.1 files are downloaded in full. `MAX_TRAIN_EXAMPLES=None`
trains on all eligible training examples; set 10,000 for a shorter pilot project.
Examples are retained only when the full formatted question and context fit
the input cap. No answer-guided cropping occurs. Internal development articles
come exclusively from training, and official validation is reserved for final
reporting. Scores describe the retained subset.

Copy the exact `DataConfig` between notebooks. Compare `data_fingerprint` values
before comparing results. You can restore `data/` from previous output to reuse
tokenization, but both notebooks can independently reproduce the same split.

In [ ]:
from qa_assignment.config import DataConfig
from qa_assignment.data import prepare_data, QACollator

DATA_CONFIG = DataConfig(
    max_input_length=512,
    max_output_length=64,
    development_fraction=0.1,
    split_seed=2026,
    max_train_examples=None,
)
bundle = prepare_data(OUTPUT_ROOT / "data", DATA_CONFIG)
collator = QACollator(bundle.tokenizer, DATA_CONFIG)
print("Data fingerprint:", bundle.manifest["data_fingerprint"])
print("Prepared cache:", bundle.directory)
print("Train / development / validation:", len(bundle.train), len(bundle.development), len(bundle.validation))
bundle.manifest["statistics"]

## Experiment configuration

Start small and compare the pilot estimates before training. All variants use
identical dimensions, feed-forward shells, positional embeddings, tokenizer,
data, and training recipe. Mamba encoders scan both directions; Mamba decoders
are causal. `mamba_mamba` still uses cross-attention.

The default runs all four variants sequentially, one on GPU at a time. Set
`MODEL_VARIANTS=['attn_attn', 'mamba_attn']` for the core scope. Use the same
precision for all variants. FP32 is the initial compatibility setting; FP16
needs a successful probe. Multiple seeds multiply training cost.

In [ ]:
from dataclasses import replace
from qa_assignment.config import ModelConfig, TrainConfig, BenchmarkConfig, VARIANTS

MODEL_VARIANTS = list(VARIANTS)
SEEDS = [42]  # extend to [42, 43, 44] if your budget permits
MODEL_CONFIG = ModelConfig(d_model=128, encoder_layers=2, decoder_layers=2,
                           heads=4, feedforward_dim=512, dropout=0.1)
TRAIN_CONFIG = TrainConfig(epochs=20, micro_batch_size=4, accumulation_steps=4,
                           eval_batch_size=8, learning_rate=3e-4, precision="fp32",
                           save_every_steps=250, development_limit=None)
BENCHMARK_CONFIG = BenchmarkConfig(examples=200, repeats=3,
                                   throughput_batch_size=4, fixed_output_tokens=32)
print("Effective batch:", TRAIN_CONFIG.micro_batch_size * TRAIN_CONFIG.accumulation_steps)
print("Variants:", MODEL_VARIANTS, "Seeds:", SEEDS)

## Compatibility and resource pilot

The pilot verifies finite training gradients and cached/uncached decoding logits,
then times 50 warmed training microsteps on a repeated small batch. Pilot models
are discarded; main runs start fresh. The epoch estimate excludes development
evaluation, checkpoints, and final generation. Inspect the memory and timing
results before proceeding. Change shared configuration before main runs if needed.

In [ ]:
from qa_assignment.runtime import run_pilot
from qa_assignment.utils import write_json

pilots = {}
for variant in MODEL_VARIANTS:
    pilots[variant] = run_pilot(bundle, replace(MODEL_CONFIG, variant=variant),
                                TRAIN_CONFIG, collator, DEVICE, steps=50, repo_root=REPO_ROOT)
    print(variant, pilots[variant])
write_json(OUTPUT_ROOT / "results" / "controlled_pilots.json", pilots)

## Optional tiny-subset learning diagnostic

Enable this to check that a fresh model can learn four fixed training examples.
This diagnostic is separate from the main experiment and final validation.
Increase steps if it has not overfit; inspect the answer examples and losses.
A successful resource probe alone does not show that a model learned QA.

In [ ]:
from qa_assignment.diagnostics import overfit_diagnostic

RUN_OVERFIT_DIAGNOSTIC = False
if RUN_OVERFIT_DIAGNOSTIC:
    for variant in MODEL_VARIANTS:
        report = overfit_diagnostic(bundle, replace(MODEL_CONFIG, variant=variant),
                                     collator, DEVICE, steps=300)
        print(variant, report)

## Train, checkpoint, final evaluation, and benchmark

Training is enabled below, as in the sample notebook. Run this cell after
reviewing the pilot results. Checkpoints are isolated by variant and seed.
Existing matching runs resume automatically; optimizer, scheduler, scaler,
random state, and within-epoch cursor are restored. Final validation runs only
after training and checkpoint selection finish.

In [ ]:
from qa_assignment.workflow import run_experiment, archive_outputs

RUN_TRAINING = True
summaries = []
if RUN_TRAINING:
    try:
        for variant in MODEL_VARIANTS:
            for seed in SEEDS:
                summaries.append(run_experiment(
                    bundle, replace(MODEL_CONFIG, variant=variant), replace(TRAIN_CONFIG, seed=seed),
                    OUTPUT_ROOT, device=DEVICE, resume="auto",
                    benchmark_config=BENCHMARK_CONFIG, repo_root=REPO_ROOT,
                ))
    finally:
        print("Available checkpoints/results archived at:", archive_outputs(OUTPUT_ROOT))
else:
    print("Training disabled; set RUN_TRAINING=True after inspecting pilots.")

## Results and output archive

Results include normalized EM/F1, predictions, selected checkpoint, resource
measurements, and warmed timing repeats. Actual QA latency and forced 32-token
decoding are separate measurements. T5 and the controlled models are plotted
separately. Import prior `results/` and `checkpoints/` to see runs from the other
notebook. Matching data fingerprints are required for comparison.

In [ ]:
from qa_assignment.workflow import collect_results, plot_results, archive_outputs

table = collect_results(OUTPUT_ROOT)
display(table)
if not table.empty:
    figures = plot_results(table, OUTPUT_ROOT)
    import matplotlib.pyplot as plt
    plt.show()

In [ ]:
INCLUDE_DATA_IN_ARCHIVE = True  # enables reuse in the second notebook
archive_path = archive_outputs(OUTPUT_ROOT, include_data=INCLUDE_DATA_IN_ARCHIVE)
print("Download from Kaggle Output:", archive_path)
print("Checkpoint root:", OUTPUT_ROOT / "checkpoints")

## Resume in another Kaggle session

Save/download these outputs, attach the prior output as a Kaggle input, and set
`RESTORE_FROM` to the folder containing `checkpoints/`, `results/`, and optionally
`data/`. Restore before preparing new artifacts. The helper copies read-only
Kaggle inputs into writable working storage; existing folders are never overwritten.
`resume='auto'` loads each variant's `last.pt`. Keep the complete checkpoint folder,
including `best.pt`, and use identical model/data/training settings and source commit.

To start a different experiment, change `OUTPUT_ROOT`; changing settings inside an
existing run is rejected. `last.pt` resumes training; `best.pt` is selected using
internal-development F1. Load only checkpoints you created or otherwise trust.